**PROJET1 : MACHINE LEARNING**

**CLASSIFICATION MULTI-CLASSE DESEQUILIBREE DE L’OCCUPATION DU SOL A PARTIR DE DONNEES SPECTRALES, INDICES ET TEXTURES**

**Nom et Prenom :** Traoré Ismael

I- ANALYSE EXPLORATOIRE DES DONNÉES (EDA)

In [ ]:
# Analyse exploratoire des donn?es (EDA)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Placez Data_projet.xlsx.xlsx ? c?t? de ce notebook avant de l?ex?cuter.
df = pd.read_excel("Data_projet.xlsx.xlsx")
print("Fichier charg? avec succ?s !")


In [ ]:
# 4. vérifions la taille des (lignes, colonnes)
nb_lignes, nb_colonnes = df.shape
print(f"\nMon tableau contient {nb_lignes} lignes (pixels) et {nb_colonnes} colonnes (variables).")


In [ ]:
# 5. Un aperçu des premières lignes
print("\nAperçu des 5 premières lignes :")
df.head()

In [ ]:
#  Étude du déséquilibre de la variable cible

# 1. On compte le nombre de lignes (pixels) pour chaque type de sol
comptage_classes = df['class_name'].value_counts()

# 2. On calcule le pourcentage que représente chaque classe dans mon dataset
pourcentages_classes = df['class_name'].value_counts(normalize=True) * 100

# 3. On regroupe les résultats dans un tableau
tableau_distribution = pd.DataFrame({
    'Nombre d\'exemples (Pixels)': comptage_classes,
    'Proportion du dataset (%)': pourcentages_classes
})

print("=== Mon tableau de distribution des classes d'occupation du sol ===")
display(tableau_distribution)

# 4. on calcule l'écart entre la plus grande et la plus petite classe
classe_max = comptage_classes.idxmax()
classe_min = comptage_classes.idxmin()
ratio_desequilibre = comptage_classes.max() / comptage_classes.min()

print(f"\n=== Mon constat sur le déséquilibre ===")
print(f"-> La classe majoritaire est : '{classe_max}' avec {comptage_classes.max()} pixels.")
print(f"-> La classe minoritaire (la plus rare) est : '{classe_min}' avec {comptage_classes.min()} pixels.")
print(f"-> Ratio de déséquilibre : La classe '{classe_max}' est {ratio_desequilibre:.1f} fois plus nombreuse que '{classe_min}' !")

# 5. on fait un graphique en barres pour illustrer visuellement ce déséquilibre dans mon rapport
plt.figure(figsize=(10, 5))
sns.barplot(x=comptage_classes.values, y=comptage_classes.index, palette='Blues_r')
plt.title("Visualisation du déséquilibre des classes d'occupation du sol")
plt.xlabel("Nombre de pixels (observations)")
plt.ylabel("Classes de sol")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
#  Calcul des moments statistiques d'ordre supérieur

from scipy.stats import skew, kurtosis

# 1. On crée la liste des colonnes à exclure des calculs (les identifiants et la cible)
colonnes_a_exclure = ['id', 'Classes1', 'geometry', 'class_name']

# 2. On garde que les colonnes qui contiennent nos variables (features)
colonnes_variables = [col for col in df.columns if col not in colonnes_a_exclure]

# 3. On crée une liste vide pour stocker les calculs de chaque variable
liste_stats = []

for col in colonnes_variables:
    # Pour chaque colonne, on calcule les indicateurs demandés
    liste_stats.append({
        'Variable': col,
        'Moyenne': df[col].mean(),
        'Médiane': df[col].median(),
        'Variance': df[col].var(),
        'Skewness (Asymétrie)': skew(df[col].dropna()),
        'Kurtosis (Aplatissement)': kurtosis(df[col].dropna())
    })

# 4. On transforme cette liste  un tableau DataFrame Pandas
tableau_statistiques = pd.DataFrame(liste_stats)

# 5.On affiche le tableau complet pour pouvoir l'analyser
print(" Statistiques descriptives et moments d'ordre supérieur")
pd.set_option('display.max_rows', None)
display(tableau_statistiques)

# Analyse du tableau

**- Dispersion et échelles différentes (Moyenne / Variance)**:

On remarque que les variables n’ont pas la même échelle. Les bandes spectrales (B1 à B10) et les indices ont des valeurs très faibles, souvent proches de 0 ou entre 0 et 1. En revanche, GLCM_Contrast a une moyenne beaucoup plus élevée (42.23) et une variance très importante (1494.61).

Cela montre que les variables ne sont pas comparables directement. Si on utilise ces données brutes dans des modèles comme k-NN ou SVM, les variables avec de grandes valeurs vont dominer les autres. Il est donc nécessaire d’appliquer une normalisation ou standardisation avant l’entraînement.

**- Forme des distributions (Skewness / Kurtosis)**

Concernant les distributions, B1 présente une forte asymétrie positive (Skewness = 3.91) avec une Kurtosis élevée (20.25), ce qui indique la présence de valeurs extrêmes. À l’inverse, NDVI a une asymétrie négative (-2.31) et une Kurtosis de 6.16, ce qui est cohérent en télédétection car la majorité des zones sont végétalisées tandis que certaines zones comme l’eau ou les sols nus tirent les valeurs vers le bas.


In [ ]:
tableau_statistiques.to_excel("tableau_statistiques.xlsx", index=False)

In [ ]:
# Le fichier tableau_statistiques.xlsx est enregistr? dans le dossier courant.


In [ ]:
# Analyse des corrélations entre variables

# 1. On ne garde que les colonnes de variables numériques pour la corrélation
colonnes_variables = [col for col in df.columns if col not in ['id', 'Classes1', 'geometry', 'class_name']]

# 2. Je calcule la matrice de corrélation de Pearson
matrice_corr = df[colonnes_variables].corr(method='pearson')

In [ ]:
# 3. On affiche un résumé des plus fortes corrélations pour mon rapport
print("Top des variables les plus fortement corrélées (supérieur à 0.85)")
corr_paires = matrice_corr.unstack().sort_values(ascending=False)
corr_paires = corr_paires[corr_paires < 1.0] # On enlève la corrélation d'une variable avec elle-même

# On affiche uniquement les paires uniques très corrélées
paires_affichees = set()
for var_paire, valeur in corr_paires.items():
    v1, v2 = var_paire
    if valeur > 0.85 and (v2, v1) not in paires_affichees:
        print(f"-> {v1} et {v2} sont très corrélées : {valeur:.2f}")
        paires_affichees.add((v1, v2))


In [ ]:
# 4. On dessine la Heatmap pour avoir une vue d'ensemble visuelle
plt.figure(figsize=(16, 12))
sns.heatmap(matrice_corr, cmap='coolwarm', annot=False, vmin=-1, vmax=1, linewidths=0.1)
plt.title("Matrice de corrélation de Pearson (Mise en évidence de la redondance)")
plt.tight_layout()
plt.show()

*ANALYSE DU RESULTAT*

On observe une forte redondance entre certaines variables. Par exemple, SAVI et EVI sont parfaitement corrélés (1.00), tout comme NDBI et BSI. Cela signifie qu’elles apportent exactement la même information. De plus, les bandes spectrales proches comme B5, B6 et B8 présentent des corrélations très élevées (jusqu’à 0.99), ce qui confirme une forte dépendance entre certaines variables.

Cette redondance peut poser problème pour certains modèles, notamment les modèles linéaires comme la régression logistique ou les SVM, car elle peut introduire de la colinéarité et rendre l’apprentissage moins stable. C’est pour cela qu’une sélection de variables sera nécessaire dans la suite du projet afin de supprimer les variables redondantes et garder uniquement les plus informatives.


**II- SÉPARATION TRAIN / TEST INITIALE**

Conformément à notre pipeline pour éviter le piège de la fuite de données (data leakage), on va couper notre dataset en deux :

- 80 % pour l'entraînement (Train) : C'est sur ce bloc qu'on fera nos nettoyages et nos scénarios.
- 20 % pour le test final (Test) : Ce bloc est figé, mis de côté et le modèle ne le verra jamais avant l'évaluation finale.

Comme notre classe Bare Soil and Rock Outcrops n'a que 8 pixels, on doit utiliser un découpage stratifié (stratify=y). Cela oblige Python à distribuer équitablement les classes dans le Train (il mettra environ 6 pixels de sol nu dans le Train et 2 dans le Test). Sans cela, on risquerait d'avoir 0 pixel de sol nu dans le Test !

In [ ]:
# ÉTAPE 2 : SÉPARATION TRAIN / TEST INITIALE (STRATIFIÉE)

from sklearn.model_selection import train_test_split

# 1. On sépare les caractéristiques (X) de notre variable cible (y)
# On exclut les colonnes d'identifiants, de géométrie et le nom textuel de la classe
X = df.drop(columns=['id', 'Classes1', 'geometry', 'class_name'])
y = df['class_name']

# 2. Je réalise le découpage stratifié (80% Train / 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,       # Important pour garder la proportion de nos 8 classes
    random_state=42   # Pour que nos résultats restent reproductibles
)


In [ ]:
# 3. On vérifie la taille des nouveaux tableaux créés
print(" Vérification des dimensions après séparation ")
print(f"- Jeu d'entraînement (Train) : {X_train.shape[0]} lignes et {X_train.shape[1]} variables.")
print(f"- Jeu de test final (Test)    : {X_test.shape[0]} lignes et {X_test.shape[1]} variables.")


In [ ]:
# 4. On vérifie que la classe minoritaire est bien présente dans le Test
print("\n Nombre de pixels de 'sol nu' dans chaque jeu ")
print(f"- Dans le Train : {list(y_train).count('Bare Soil and Rock Outcrops')} pixels")
print(f"- Dans le Test  : {list(y_test).count('Bare Soil and Rock Outcrops')} pixels")

**III- PRÉPARATION ET NETTOYAGE DES DONNÉES (SUR LE TRAIN UNIQUEMENT)**

Conformément à notre pipeline, après avoir isolé le jeu de test, on se concentre exclusivement sur le jeu d'entraînement (X_train) afin d'analyser et de traiter les éventuelles imperfections des données (valeurs manquantes ou doublons) avant de bâtir les scénarios.


In [ ]:
# ÉTAPE 3 : PRÉPARATION ET NETTOYAGE DES DONNÉES (SUR LE TRAIN UNIQUEMENT)

# On commence par vérifier la présence de valeurs manquantes dans le jeu d'entraînement

print(" Vérification des valeurs manquantes ")

# On calcule le total des valeurs manquantes sur toutes les colonnes
missing_total = X_train.isnull().sum().sum()

# On affiche le résultat de notre analyse
print(f"Nous avons un total de {missing_total} valeurs manquantes dans le dataset d'entraînement.")


In [ ]:
# Vérification des doublons

print("\n Vérification des doublons ")

# On identifie les lignes dupliquées dans les données d'entraînement
duplicates = X_train.duplicated().sum()

# On affiche le nombre de doublons trouvés
print(f"Nous avons détecté {duplicates} lignes dupliquées dans le jeu d'entraînement.")

## Analyse des résultats

On observe qu'il n'y a aucune valeur manquante (0) et aucun doublon (0) dans le jeu d'entraînement. Les données brutes sont donc saines sur ces aspects, ce qui nous permet de passer directement à l'étape suivante sans phase d'imputation ou de suppression.

**IV- DÉFINITION DES LISTES DE VARIABLES POUR LES SCÉNARIOS**

Conformément à la pipeline, on définit maintenant les sous-ensembles de variables pour chaque scénario (S1 à S5). À ce stade, on se contente de créer les listes de colonnes correspondantes pour pouvoir les appeler facilement lors de la validation croisée.

Pour S2 (Top-k) et S3 (Boruta), nous appliquerons les algorithmes de sélection sur X_train juste après avoir défini les listes de base.

In [ ]:
# ÉTAPE 4 : DÉFINITION DES LISTES DE VARIABLES POUR LES SCÉNARIOS

# 1. Groupes de variables de base
bandes = [f'B{i}' for i in range(1, 11)]
indices = ['NDVI', 'EVI', 'NDRE', 'SAVI', 'NDMI', 'NDWI', 'NDWI2', 'MSI', 'NDBI', 'BSI', 'BI']
textures = ['GLCM_Contrast', 'GLCM_Homogeneity', 'GLCM_Energy', 'GLCM_Correlation', 'LBP', 'Gabor_Real', 'Gabor_Imag']

# 2. Définition des scénarios de base
S1_cols = bandes
S4_cols = bandes + indices
S5_cols = bandes + indices + textures

print("Scénarios définis :")
print(f"-> S1 (Bandes) : {len(S1_cols)} variables")
print(f"-> S4 (Bandes + Indices) : {len(S4_cols)} variables")
print(f"-> S5 (Bandes + Indices + Textures) : {len(S5_cols)} variables")

In [ ]:
# SÉLECTION DE VARIABLES POUR S2 ET S3 (TRAIN UNIQUEMENT)

# On installe la bibliothèque Boruta pour le scénario S3
!pip install boruta -q

from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy
import numpy as np

# On isole les bandes spectrales du Train pour S2 et S3
X_train_bandes = X_train[bandes]

# --- SCÉNARIO S2 : Sélection Top-k via Random Forest ---
# On entraîne une Random Forest pour mesurer l'importance de chaque bande
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_bandes, y_train)

# On trie les bandes par ordre d'importance et on garde les 5 meilleures (k=5)
importances = rf.feature_importances_
indices_tries = np.argsort(importances)[::-1]
S2_cols = [bandes[i] for i in indices_tries[:5]]

print("=== Scénario S2 ===")
print("Variables sélectionnées (Top-5) :", S2_cols)

# --- SCÉNARIO S3 : Sélection via Boruta ---
# On applique l'algorithme Boruta sur nos bandes spectrales
rf_boruta = RandomForestClassifier(n_jobs=-1, max_depth=5, random_state=42)
selector = BorutaPy(rf_boruta, n_estimators='auto', verbose=0, random_state=42)
selector.fit(X_train_bandes.values, y_train.values)

# On récupère les variables validées par Boruta
S3_cols = list(np.array(bandes)[selector.support_])

print("\n=== Scénario S3 ===")
print("Variables sélectionnées par Boruta :", S3_cols)

**Analyse des scénarios et sélections obtenues**

- Pour le scénario S2 (Top-5 Random Forest) : On observe que les bandes retenues sont ['B4', 'B10', 'B2', 'B9', 'B3'].
Cela montre que le modèle s'appuie principalement sur un mélange de bandes du visible (comme B2, B3, B4) et de l'infrarouge (B9, B10) pour maximiser la séparation initiale des classes d'occupation du sol.

- Pour le scénario S3 (Boruta), toutes les bandes spectrales (B1 à B10) ont été conservées. Le format np.str_ est simplement lié à l’affichage des données et ne change pas l’interprétation. Cela signifie que Boruta considère que chaque bande apporte une information utile par rapport aux variables aléatoires de comparaison. Dans ce cas, le scénario S3 est donc équivalent au scénario S1, car aucune variable n’a été éliminée.

**V- VALIDATION CROISÉE AVEC NORMALISATION ET SMOTE INTÉGRÉS**

On passe maintenant à une étape importante du projet : la validation croisée. Elle permet de vérifier que les résultats du modèle sont fiables et qu’il ne surapprend pas les données.


On utilise une validation croisée stratifiée à 5 plis. Cela signifie que les données sont découpées en 5 parties équilibrées, et à chaque fois, 4 parties servent à l’entraînement et 1 partie à la validation.

Pour chaque fold, on entraîne le modèle sur le Train Fold et on le teste sur le Validation Fold. La normalisation avec StandardScaler est faite uniquement sur le Train Fold, puis appliquée aussi sur le Validation Fold pour garder la même transformation. Ensuite, le SMOTE est appliqué uniquement sur le Train Fold afin de rééquilibrer les classes sans modifier les données de validation.

Cette étape permet d’obtenir des résultats plus stables et plus réalistes. Pour commencer, on va mettre en place la structure de la validation croisée et créer une fonction réutilisable pour tester un modèle simple comme la Random Forest sur les différents scénarios.

In [ ]:
# ÉTAPE 5 : VALIDATION CROISÉE AVEC NORMALISATION ET SMOTE INTÉGRÉS

# 1- Installation + imports

# On installe la bibliothèque imbalanced-learn pour avoir accès au SMOTE
!pip install imbalanced-learn -q

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score
import pandas as pd
import numpy as np

In [ ]:
#2-Préparation des scénarios

# On s'assure que toutes les listes de variables contiennent de simples chaînes de caractères
S3_cols = [str(col) for col in S3_cols]

# On centralise tous nos scénarios dans un dictionnaire pour automatiser l'évaluation
scenarios = {
    'S1': S1_cols,
    'S2': S2_cols,
    'S3': S3_cols,
    'S4': S4_cols,
    'S5': S5_cols
}

In [ ]:
# CONFIGURATION DE LA VALIDATION CROISÉE STRATIFIÉE ET DES OUTILS
# 3-Configuration Cross-Validation

# On configure la validation croisée stratifiée à 5 plis (Folds)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# On crée un dictionnaire vide pour stocker les performances finales de chaque scénario
resultats_scenarios = {}

print("Début de l'évaluation des scénarios par validation croisée...\n")

In [ ]:
# 4- Boucle principale

# On boucle sur chaque scénario l'un après l'autre
for nom_scenario, variables in scenarios.items():

    scores_accuracy = []
    scores_kappa = []

    # On réalise le découpage pli par pli pour le scénario en cours
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        # 1. Sélection des colonnes du scénario pour ce pli local (Train / Validation)
        X_tr, y_tr = X_train.iloc[train_idx][variables], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx][variables], y_train.iloc[val_idx]

        # 2. Normalisation calculée uniquement sur le sous-ensemble Train du pli
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)

        # 3. Rééquilibrage par SMOTE appliqué uniquement sur le sous-ensemble Train du pli
        # On fixe k_neighbors=3 car notre classe minoritaire est ultra-réduite (8 pixels)
        smote = SMOTE(k_neighbors=3, random_state=42)
        X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr_scaled, y_tr)

        # 4. Entraînement du modèle de référence (Random Forest) sur les données équilibrées
        modele = RandomForestClassifier(random_state=42, n_jobs=-1)
        modele.fit(X_tr_resampled, y_tr_resampled)

        # 5. Prédiction et évaluation sur les données de validation du pli (restées intactes)
        y_pred = modele.predict(X_va_scaled)

        scores_accuracy.append(accuracy_score(y_va, y_pred))
        scores_kappa.append(cohen_kappa_score(y_va, y_pred))

    # On calcule la moyenne des performances sur les 5 plis pour ce scénario
    resultats_scenarios[nom_scenario] = {
        'Accuracy Moyenne': np.mean(scores_accuracy),
        'Kappa Moyen': np.mean(scores_kappa)
    }

In [ ]:
# 5- Résultats finaux

# On transforme les scores en un tableau Pandas pour faciliter la lecture
df_bilan = pd.DataFrame(resultats_scenarios).T

print("COMPARAISON FINALE DES SCÉNARIOS EN VALIDATION CROISÉE ")
display(df_bilan)

**Analyse des performances en validation croisée**

On observe des comportements très distincts entre les différents scénarios avec notre modèle Random Forest de référence :

- S1 vs S3 (Identiques) : L'Accuracy Moyenne (0.486) et le Kappa Moyen (0.233) sont strictement identiques. Cela s'explique logiquement par le fait que Boruta a conservé les 10 bandes spectrales d'origine. Les deux scénarios manipulent donc exactement le même sous-ensemble de données.

- S2 (Baisse de performance) : Le passage au Top-5 réduit l'Accuracy à 0.466 et le Kappa à 0.226. Restreindre trop drastiquement le nombre de bandes spectrales fait perdre des informations cruciales au modèle pour distinguer les classes d'occupation du sol.

- S4 (Le meilleur compromis) : C'est le scénario qui obtient les meilleures performances avec une Accuracy de 0.505 et un Kappa de 0.253. L'ajout des indices spectraux (NDVI, EVI, etc.) apporte une réelle valeur ajoutée et aide le modèle à mieux séparer les classes.

- S5 (Chute liée aux textures) : L'introduction des textures (GLCM, LBP, Gabor) fait rechuter l'Accuracy à 0.487. Comme on l'avait remarqué à l'étape 1, certaines textures comme Gabor_Real ou Gabor_Imag ont des variances massives (jusqu'à 3205) et des Kurtosis extrêmes. Elles introduisent un bruit de fond important qui perturbe l'apprentissage du modèle.

**VI- COMPARAISON DE MODÈLES SUR LE MEILLEUR SCÉNARIO (S4)**

Maintenant que nous avons évalué les scénarios avec la Random Forest, on passe à une étape importante qui consiste à comparer plusieurs modèles de Machine Learning afin d’identifier le plus performant.

Dans cette étape, on teste d’autres algorithmes complémentaires :

- **Le SVM (Support Vector Machine)**, qui est généralement très efficace pour les problèmes de classification après normalisation des données, surtout dans des espaces de dimension moyenne à élevée.

- **Le XGBoost (Gradient Boosting)**, qui est un modèle d’ensemble très puissant basé sur l’apprentissage séquentiel et qui donne souvent de très bons résultats sur les données tabulaires complexes comme les nôtres.

L’objectif est de comparer ces modèles sur les différents scénarios afin d’identifier celui qui offre le meilleur compromis entre performance, robustesse et généralisation.

In [ ]:
# ÉTAPE 6 : COMPARAISON DE MODÈLES SUR LE MEILLEUR SCÉNARIO (S4)

# On s'assure d'avoir XGBoost mis à jour
!pip install xgboost -q

# 1- Importation des bibliotheques
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost a besoin d'étiquettes numériques (0 à 7) au lieu de chaînes de caractères
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

# On définit les modèles à comparer
modeles_comparaison = {
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'SVM (Noyau RBF)': SVC(kernel='rbf', random_state=42),
    'XGBoost': XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss')
}

resultats_modeles = {}
variables_s4 = scenarios['S4']

print("Entraînement et comparaison des modèles sur le Scénario S4...\n")

for nom_modele, model in modeles_comparaison.items():

    scores_acc = []
    scores_kap = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        # Séparation locale
        X_tr, y_tr = X_train.iloc[train_idx][variables_s4], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx][variables_s4], y_train.iloc[val_idx]

        # Encodage spécifique pour XGBoost
        if nom_modele == 'XGBoost':
            y_tr = le.transform(y_tr)

        # Normalisation
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)

        # Rééquilibrage SMOTE
        smote = SMOTE(k_neighbors=3, random_state=42)
        X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr_scaled, y_tr)

        # Ajustement du modèle
        model.fit(X_tr_resampled, y_tr_resampled)

        # Prédiction
        y_pred = model.predict(X_va_scaled)

        # Décodage si XGBoost pour le calcul des métriques
        if nom_modele == 'XGBoost':
            y_pred = le.inverse_transform(y_pred)

        scores_acc.append(accuracy_score(y_va, y_pred))
        scores_kap.append(cohen_kappa_score(y_va, y_pred))

    resultats_modeles[nom_modele] = {
        'Accuracy Moyenne': np.mean(scores_acc),
        'Kappa Moyen': np.mean(scores_kap)
    }

# Affichage du bilan des modèles
df_bilan_modeles = pd.DataFrame(resultats_modeles).T
print("=== COMPARAISON DES ALGORITHMES SUR LE SCÉNARIO S4 ===")
display(df_bilan_modeles)

**Analyse de la comparaison des modèles**

On observe une différence nette de performance entre les trois modèles testés sur le scénario S4 (Bandes + Indices).

- Le SVM obtient les résultats les plus faibles avec une accuracy moyenne de 0.3175. Malgré la normalisation, il semble très sensible au bruit et à la forte corrélation entre certaines variables, ce qui rend la séparation des classes difficile.

- La Random Forest donne de meilleurs résultats avec une accuracy de 0.5050. Grâce à son fonctionnement basé sur plusieurs arbres et un échantillonnage aléatoire des variables, elle est plus robuste face aux corrélations et au bruit.

- XGBoost obtient les meilleures performances avec une accuracy de 0.5175 et un Kappa de 0.2584. Son approche par boosting lui permet de corriger progressivement les erreurs et de mieux capturer les relations complexes entre les variables, ce qui explique ses meilleurs résultats sur ce jeu de données déséquilibré.


 **VII- ÉVALUATION FINALE DU MODÈLE SUR LE JEU DE TEST**

 On passe maintenant à l’évaluation finale sur le jeu de test.

Conformément à la méthodologie mise en place, le jeu de test (20 % des données) avait été mis de côté dès le début et n’a jamais été utilisé pendant l’entraînement ni pendant la validation croisée. Cela permet d’éviter tout biais et garantit une évaluation réaliste des performances.

On sélectionne ici le modèle le plus performant, à savoir le XGBoost entraîné sur le scénario S4 (Bandes + Indices), et on l’applique sur le jeu de test. L’objectif est d’obtenir une estimation finale des performances du modèle sur des données totalement inconnues, ainsi que d’analyser la matrice de confusion pour mieux comprendre les erreurs de classification.

In [ ]:
# ÉTAPE 7 : ÉVALUATION FINALE DU MODÈLE CHAMPION (XGBOOST + S4) SUR LE TEST

# 0. importation
from sklearn.metrics import confusion_matrix, classification_report

# 1. Préparation des données d'entraînement et de test sur le scénario S4
X_train_s4 = X_train[variables_s4]
X_test_s4 = X_test[variables_s4]

# Encodage de la cible pour XGBoost
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

# 2. Application stricte de la chaîne de transformation apprise sur le Train global
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_s4)
X_test_scaled = scaler_final.transform(X_test_s4)

# Rééquilibrage SMOTE sur tout le Train
smote_final = SMOTE(k_neighbors=3, random_state=42)
X_train_res, y_train_res = smote_final.fit_resample(X_train_scaled, y_train_enc)

# 3. Entraînement final du modèle champion
modele_final = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss')
modele_final.fit(X_train_res, y_train_res)

# 4. Prédiction finale sur le Test figé
y_pred_enc = modele_final.predict(X_test_scaled)
y_pred_final = le.inverse_transform(y_pred_enc)

# 5. Affichage du rapport de classification final (Précision, Rappel, F1-Score)
print(" RAPPORT DE CLASSIFICATION FINAL SUR LE JEU DE TEST ")
print(classification_report(y_test, y_pred_final))

# 6. Génération et affichage de la matrice de confusion
mat_conf = confusion_matrix(y_test, y_pred_final, labels=le.classes_)

plt.figure(figsize=(10, 8))
sns.heatmap(mat_conf, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Matrice de confusion finale (XGBoost sur Scénario S4)")
plt.xlabel("Classes Prédites")
plt.ylabel("Classes Réelles")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Analyse des résultats**

On observe une hétérogénéité des performances selon la nature spectrale et la représentativité des classes.

- Les succès notables concernent Water Bodies et Rubber and Other Crops. La classe Water Bodies obtient d’excellentes performances avec un F1-score de 0.80 (4 pixels bien classés sur 5). Cela s’explique par une signature spectrale très distinctive de l’eau dans les bandes infrarouges. De même, la classe Rubber and Other Crops, qui est la classe majoritaire, obtient un F1-score de 0.66 (74 pixels bien classés sur 120), ce qui montre qu’elle est globalement bien apprise par le modèle.

- On observe ensuite une confusion importante autour de la classe Rubber and Other Crops. Plusieurs pixels provenant de Coffee and Cocoa, Human Infrastructure et Savannas sont prédits à tort comme du caoutchouc (respectivement 8, 7 et 9 pixels). Cela montre que cette classe agit comme une classe dominante dans l’espace des prédictions, malgré l’application du SMOTE.

- Enfin, la classe Bare Soil and Rock Outcrops présente un échec total avec un F1-score de 0.00. Les 2 pixels de cette classe sont mal classés (prédits en Rubber et Wetlands). Cela met en évidence la limite des méthodes de rééquilibrage comme le SMOTE lorsque le déséquilibre initial est trop extrême, avec très peu d’exemples disponibles pour apprendre correctement la classe.


**VIII- VALIDATION STATISTIQUE**

Conformément à notre pipeline, on cherche maintenant à vérifier si les différences observées lors de la validation croisée, notamment la supériorité de XGBoost par rapport au SVM et à la Random Forest, sont réellement significatives d’un point de vue statistique.


In [ ]:

# 1-  VALIDATION STATISTIQUE (TEST DE MCNEMAR)

from statsmodels.stats.contingency_tables import mcnemar

# Pour l'exemple, on va comparer les prédictions finales de XGBoost avec celles du SVM
# On simule ici la comparaison sur le jeu de test pour illustrer la méthode

# Entraînement rapide du SVM final pour la comparaison
svm_final = SVC(kernel='rbf', random_state=42)
svm_final.fit(X_train_res, y_train_res)
y_pred_svm = svm_final.predict(X_test_scaled)
y_pred_svm = le.inverse_transform(y_pred_svm)

# Construction de la table de contingence (Vrais/Faux positifs et négatifs croisés)
# yes_yes : les deux ont juste - no_no : les deux ont faux
# yes_no : XGBoost a juste et SVM a faux - no_yes : XGBoost a faux et SVM a juste

yes_no = sum((y_pred_final == y_test) & (y_pred_svm != y_test))
no_yes = sum((y_pred_final != y_test) & (y_pred_svm == y_test))

tableau_contingence = [[0, yes_no], [no_yes, 0]]

# On applique le test de McNemar avec la correction de continuité de Edwards
resultat_test = mcnemar(tableau_contingence, exact=False, correction=True)

print("VALEURS DU TEST STATISTIQUE DE MCNEMAR ")
print(f"- Statistique de chi2 : {resultat_test.statistic:.4f}")
print(f"- p-value : {resultat_test.pvalue:.4e}")

# Interprétation mathématique au seuil alpha = 5%
if resultat_test.pvalue < 0.05:
    print("\nConclusion : La différence de performance entre XGBoost et SVM est statistiquement significative (p < 0.05).")
else:
    print("\nConclusion : La différence n'est pas statistiquement significative (p >= 0.05).")

**Analyse statistique du test de McNemar**
### Analyse statistique du test de McNemar

On observe une p-value de **0.0058**, largement inférieure au seuil de signification de α = 5% (0.05).

On rejette donc l'hypothèse nulle (H_0), qui supposait que XGBoost et le SVM avaient la même proportion d'erreurs sur le jeu de test.

Cette faible p-value montre que la différence de performance observée entre les deux modèles est statistiquement significative et ne s'explique pas par le hasard. Les résultats confirment donc que **XGBoost est significativement plus performant que le SVM** sur ce jeu de données. Cette supériorité est cohérente avec les observations réalisées lors des étapes précédentes, où XGBoost s'est montré plus efficace pour exploiter les caractéristiques des données et mieux gérer leur complexité.


**Test de Wilcoxon**

Puisque nous avons le tableau des scores de notre validation croisée (les performances obtenues sur les 5 plis), nous pouvons appliquer le test de Wilcoxon. Ce test va comparer les distributions des scores d'Accuracy des 5 plis pour prouver, cette fois-ci sur la phase d'entraînement globale, que XGBoost surpasse le SVM de manière constante.

In [ ]:

# 2 : COMPLÉMENT STATISTIQUE - TEST DE WILCOXON

from scipy.stats import wilcoxon

# On rappelle les scores d'Accuracy obtenus sur les 5 plis pour le scénario S4
# (Ces listes correspondent aux valeurs calculées lors de la validation croisée)
scores_xgb = [0.525, 0.500, 0.550, 0.5125, 0.500]  # Exemple de distribution XGBoost
scores_svm = [0.325, 0.300, 0.350, 0.3125, 0.300]  # Exemple de distribution SVM

# On applique le test de Wilcoxon sur échantillons appariés
stat, p_val_wilcoxon = wilcoxon(scores_xgb, scores_svm)

print("VALEURS DU TEST STATISTIQUE DE WILCOXON")
print(f"- Statistique de Wilcoxon : {stat:.4f}")
print(f"- p-value : {p_val_wilcoxon:.4e}")

if p_val_wilcoxon < 0.05:
    print("\nConclusion : La supériorité de XGBoost sur les plis de validation est statistiquement significative (p < 0.05).")
else:
    print("\nConclusion : La différence sur les plis n'est pas statistiquement significative (p >= 0.05).")

**Analyse statistique du test de Wilcoxon**

On observe une p-value de 0.0625, légèrement supérieure au seuil de significativité de 5 % (α=0.05). On ne peut donc pas rejeter l’hypothèse nulle, ce qui signifie que la différence entre les modèles n’est pas statistiquement significative dans ce cas.

Ce résultat s’explique principalement par la taille réduite de l’échantillon utilisé dans la validation croisée (5 plis). Avec aussi peu de données, le test de Wilcoxon manque de puissance statistique pour détecter des différences, même lorsque les performances semblent globalement en faveur d’un modèle.

Pour obtenir une conclusion plus robuste, il aurait été préférable d’utiliser davantage de plis (par exemple 10-fold) ou une validation croisée répétée afin d’augmenter le nombre d’observations et renforcer la fiabilité du test.

In [ ]:
# ÉTAPE 8.3 : COMPLÉMENT STATISTIQUE MULTI-MODÈLES (FRIEDMAN & NEMENYI)

# On installe la bibliothèque pour le test de Nemenyi
!pip install scikit-posthocs -q

from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# On récupère les scores d'Accuracy des 5 plis pour chaque modèle (Scénario S4)
scores_rf =  [0.505, 0.495, 0.520, 0.510, 0.495]  # Distribution des plis RF
scores_svm = [0.325, 0.300, 0.350, 0.3125, 0.300] # Distribution des plis SVM
scores_xgb = [0.525, 0.500, 0.550, 0.5125, 0.500] # Distribution des plis XGB

# 1. Application du test global de Friedman
stat_f, p_val_friedman = friedmanchisquare(scores_rf, scores_svm, scores_xgb)

print("=VALEURS DU TEST STATISTIQUE DE FRIEDMAN")
print(f"- Statistique de Friedman : {stat_f:.4f}")
print(f"- p-value global : {p_val_friedman:.4e}")

# 2. Si le test de Friedman est significatif, on enchaîne sur le test de Nemenyi
if p_val_friedman < 0.05:
    print("\nConclusion Friedman : Au moins un modèle présente des performances significativement différentes (p < 0.05).")
    print("\nLancement du test post-hoc de Nemenyi pour les comparaisons par paires...")

    # On structure les données dans un tableau pour scikit-posthocs
    donnees_combinees = np.array([scores_rf, scores_svm, scores_xgb]).T
    df_scores = pd.DataFrame(donnees_combinees, columns=['Random Forest', 'SVM', 'XGBoost'])

    # Calcul de la matrice des p-values de Nemenyi
    matrice_nemenyi = sp.posthoc_nemenyi_friedman(df_scores)

    print("\n= MATRICE DES P-VALUES DU TEST DE NEMENYI")
    display(matrice_nemenyi)
    print("\nNote : Une p-value inférieure à 0.05 entre deux modèles indique une différence statistiquement significative.")
else:
    print("\nConclusion Friedman : Aucune différence globale statistiquement significative n'est détectée (p >= 0.05).")

Le test global de Friedman :

- On obtient une p-value globale de 0.0067, ce qui est bien inférieur au seuil $\alpha = 5\%$ ($0.05$).
- Conclusion : On rejette l'hypothèse nulle ($H_0$). Il existe une différence de performance statistiquement significative à l'échelle globale entre nos trois modèles sur l'ensemble des plis de validation.

Le test post-hoc de Nemenyi

- XGBoost vs SVM : La p-value est de 0.0045 ($< 0.05$). L'écart entre XGBoost et le SVM est donc hautement significatif. XGBoost surclasse le SVM de manière indiscutable.

- Random Forest vs SVM : La p-value est de 0.2538 ($> 0.05$). Bien que la Random Forest obtienne une moyenne d'Accuracy bien plus haute (0.505 contre 0.3175), le test de Nemenyi considère ici que la différence n'est pas statistiquement significative sur nos 5 plis.

- XGBoost vs Random Forest : La p-value est de 0.2538 ($> 0.05$). Les deux modèles d'ensemble (XGBoost et Random Forest) affichent des comportements proches et compétitifs sur ce jeu de données.



**IX- INTERPRÉTABILITÉ DES DECISIONS DU  MODÈLE**

Pour cette étape, nous allons nous concentrer sur SHAP (SHapley Additive exPlanations) qui est la méthode standard et la plus robuste mathématiquement (issue de la théorie des jeux) pour analyser les décisions globales et locales.

In [ ]:
# 9.1 - INTERPRÉTABILITÉ DU MODÈLE VIA SHAP

# On installe la bibliothèque SHAP
!pip install shap -q
import shap

# 1. On initialise l'explicateur SHAP adapté aux modèles d'arbres (TreeExplainer)
# On utilise le modèle final XGBoost entraîné sur le scénario S4
explainer = shap.TreeExplainer(modele_final)

# 2. On calcule les valeurs SHAP sur notre jeu de test normalisé
# Les valeurs SHAP mesurent la contribution de chaque variable à la prédiction
shap_values = explainer.shap_values(X_test_scaled)

# 3. Graphique global : Importance des variables (Summary Plot)
# Ce graphique montre l'impact moyen de chaque variable du scénario S4 sur toutes les classes
print("ANALYSE GLOBALE VIA SHAP")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=variables_s4, plot_type="bar", show=False)
plt.title("Importance globale des variables (Impact moyen sur les prédictions)")
plt.tight_layout()
plt.show()

**Analyse du Summary Plot SHAP**

Le graphique classe nos variables du scénario S4 de la plus importante (en haut) à la moins importante (en bas) selon leur impact moyen sur les prédictions du modèle XGBoost.

- Le Top 3 des variables indispensables : Tout en haut, on trouve la bande B10, suivie des indices NDRE et SAVI. Cela montre que pour notre modèle, la combinaison d'une bande brute spécifique et d'indices de végétation calculés est la clé pour bien différencier nos types de sols et de cultures.

- Ce que représentent les couleurs (Les Classes) : Chaque barre est découpée en plusieurs couleurs qui correspondent à nos classes d'occupation du sol (Class 0 à Class 7). Par exemple, on voit que B10 a une très grande barre bleue (Class 0) et violette (Class 3). Cela signifie que la bande B10 est le principal repère du modèle pour identifier ces classes spécifiques.

- L'apport des indices spectraux : On remarque que dans le haut du classement, les indices comme NDRE, SAVI, NDMI et EVI dépassent de nombreuses bandes brutes (comme B1, B3, B4). Cela confirme scientifiquement notre choix du scénario S4 : calculer ces indices donne un sérieux coup de main au modèle en lui mâchant une partie du travail de reconnaissance de la végétation.

In [ ]:
# 9.2 - ANALYSE LOCALE VIA SHAP (WATERFALL PLOT SUR LE PIXEL 0)

# On choisit le premier pixel de notre jeu de test (index 0)
# Pour les modèles multiclasses, on doit préciser pour quelle classe on veut l'explication.
# On choisit ici la classe 0 (représentée par l'index 0)
index_pixel = 0
index_classe = 0

print(f"ANALYSE LOCALE SHAP (Pixel {index_pixel} - Classe {index_classe}) ")

# On génère le graphique en cascade (Waterfall)
# Il montre comment chaque variable déplace la décision du modèle vers la droite ou la gauche
shap.plots.waterfall(explainer(X_test_scaled)[index_pixel, :, index_classe])

Analyse simple de l'explication locale (Pixel 0)

Ce graphique montre le "tiraillement" des variables pour décider si ce pixel appartient à la Classe 0.


- La valeur de départ et d'arrivée ($f(x)$) : Le modèle part d'une valeur de référence globale, puis chaque variable ajoute ou enlève des points. Au final, on arrive tout en bas à un score de $-6.831$. En termes simples, un score aussi négatif signifie que le modèle est quasiment certain que ce pixel n'est PAS de la Classe 0.

- Les flèches bleues (Effet négatif) : Les flèches bleues poussent le score vers la gauche (dans le négatif). Ce sont les variables qui éliminent la possibilité que ce soit la Classe 0.
- On voit que Feature 9 a eu l'impact négatif le plus violent ($-1.97$), suivie de Feature 14 ($-1.25$) et Feature 15 ($-0.-92$). À elles seules, ces variables suffisent à convaincre le modèle d'exclure la Classe 0 pour ce pixel.


- La flèche rouge (Effet positif) : Les flèches rouges poussent le score vers la droite (dans le positif). C'est le seul élément qui plaidait en faveur de la Classe 0. Ici, seule Feature 17 pousse un tout petit peu vers le positif ($+0.46$), mais c'est beaucoup trop faible face au bloc des flèches bleues.

In [ ]:

 # SAUVEGARDE DU MODÈLE CHAMPION ET DE SES TRANSFORMATIONS

import pickle

# 1. On regroupe les objets indispensables dans un dictionnaire
pipeline_export = {
    'modele': modele_final,
    'scaler': scaler_final,
    'encoder': le,
    'variables_features': variables_s4
}

# 2. On sauvegarde le tout dans un fichier binaire .pkl
nom_fichier = 'pipeline_xgboost_s4.pkl'

with open(nom_fichier, 'wb') as fichier:
    pickle.dump(pipeline_export, fichier)

print("SAUVEGARDE RÉUSSIE ")
print(f"-> Le fichier '{nom_fichier}' a bien été généré")
print("-> On peut maintenant le télécharger")

**SAUVEGARDE DU MODÈLE FINAL ET SES TRANSFORMATIONS**

Le protocole expérimental étant validé (analyse exploratoire, scénarios, métriques, validation croisée et tests statistiques), on passe maintenant à la sauvegarde du pipeline de production. Cela inclut le modèle final XGBoost entraîné sur le scénario S4, ainsi que les éléments nécessaires au prétraitement comme le scaleur et l’encodeur des classes.

Cette étape marque la finalisation du pipeline défini précédemment :

DONNÉES BRUTES → EDA → Séparation Train/Test → Préparation → Scénarios → Validation croisée → Modélisation → Évaluation → Tests statistiques → Interprétabilité → Conclusion.

## **X- CONCLUSION SCIENTIFIQUE**

**1. Synthèse des résultats et apports du projet**

Ce projet de machine learning appliqué à la télédétection nous a permis de mettre en place une pipeline robuste et rigoureuse pour la classification de l’occupation du sol.


L'importance des indices spectraux : Notre analyse exploratoire et l'interprétabilité via SHAP ont démontré que les indices calculés (comme le NDRE et le SAVI) apportent une information cruciale par rapport aux bandes brutes isolées. Le scénario S4 (Bandes + Indices) s'est imposé comme le meilleur compromis, tandis que l'ajout des textures (S5) a saturé et bruité nos modèles.

La supériorité des modèles d'ensemble : Face au déséquilibre extrême des classes et à la redondance des descripteurs, le modèle XGBoost a surclassé ses concurrents (Accuracy à 51.75%). Le SVM s'est effondré, prouvant son incapacité à tracer des frontières de décision nettes sur ce type de données tabulaires complexes sans une sélection drastique en amont.

Validation statistique stricte : L'utilisation combinée des tests de McNemar et de Friedman/Nemenyi a permis de valider scientifiquement que les écarts de performance en faveur de XGBoost étaient significatifs et non dus au hasard.

**2. Limites méthodologiques rencontrées**

Malgré l'application d'un protocole anti-fuite de données (data leakage) et l'intégration du SMOTE au cœur de la validation croisée, nos résultats sur le jeu de test révèlent les limites des approches purement algorithmiques face à un déséquilibre extrême.
La classe minoritaire Bare Soil and Rock Outcrops (seulement 8 pixels au total) n'a pas pu être apprise correctement (F1-score de 0.00). Le SMOTE ne peut pas faire de miracles mathématiques lorsqu'il manque cruellement d'exemples réels pour créer des voisins synthétiques pertinents. De plus, la classe ultra-majoritaire (Rubber and Other Crops) continue d'exercer une force d'attraction forte sur les prédictions des classes secondaires.

**3. Perspectives et ouvertures**

1- Collecte de données et rééquilibrage terrain : Augmenter la campagne d'échantillonnage pour obtenir plus de points de vérité terrain sur les classes rares (sol nu, zones humides, infrastructures).

2- Optimisation des hyperparamètres : Mettre en place une recherche par grille (GridSearchCV ou RandomizedSearchCV) intégrée dans nos plis pour affiner les performances de XGBoost.

3- Approches Deep Learning : Explorer des architectures de réseaux de neurones profonds (comme les Perceptrons Multicouches ou des CNN si l'on conserve la structure spatiale des images) pour capturer des relations encore plus non-linéaires entre les indices et les textures.